# Task 2A - 02 Target construction
Defines **exactly** what the model learns from, and why. Weekly totals are built bottom-up from a **daily operating-day panel**, so that holidays and short weeks are structural, not noise.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2a')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
from task2a_common import *
from task2a_features import *
from task2a_models import *
from task2a_validation import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (INTERIM, METRICS, FIGURES): d.mkdir(parents=True, exist_ok=True)
o, cal = load_orders(); cal2 = enrich_calendar(cal); panel = build_panel(o, cal2)


## 1. Target definition
| Target | Definition |
|---|---|
| `vol` (daily) | sum of `order_volume_m3` of all orders (any `dispatch_status`) with `order_date` = that day, per depot x brand |
| `ch` (daily) | same, only `temp_requirement == chilled` (Fresh only) |
| weekly target | sum of the daily values over the operating days of the ISO week (`iso_year`, `iso_week` from `calendar.csv`) |

**Why `order_date`, not `dispatch_date`?** Deferred orders are dispatched later, but they were demanded on `order_date` (booklet: "assign each order to the week the store requested it").

In [2]:
chk = o.assign(od=o.date, dd=pd.to_datetime(o.dispatch_date))
cw = cal[['date', 'iso_year', 'iso_week']].assign(k=lambda x: x.iso_year * 100 + x.iso_week).set_index('date').k
chk['k_order'] = chk.od.map(cw); chk['k_dispatch'] = chk.dd.map(cw)
has = chk.dispatch_date.notna()
print('orders whose dispatch falls in a DIFFERENT ISO week than the order:', int((chk.k_order[has] != chk.k_dispatch[has]).sum()), 'of', int(has.sum()))
print(chk.groupby('dispatch_status').apply(lambda g: pd.Series({'n': len(g), 'vol_m3': g.order_volume_m3.sum()})).round(1))

orders whose dispatch falls in a DIFFERENT ISO week than the order: 215 of 96908
                       n    vol_m3
dispatch_status                   
attempted        95275.0  198665.5
deferred          1633.0    2867.1
not_run            413.0     834.6


Using `dispatch_date` (or dropping `deferred`/`not_run`) would move or remove demand: the `deferred` and `not_run` orders in the table above still represent demand and are kept on their requested week.

## 2. Chilled only exists for Fresh

In [3]:
print(o.groupby(['brand', 'temp_requirement']).order_volume_m3.sum().round(0).unstack().fillna(0))
assert o[(o.brand != 'Fresh') & (o.temp_requirement == 'chilled')].empty
f = panel[panel.brand == 'Fresh']
print('\nchilled share of Fresh volume by depot:', (f.groupby('depot').ch.sum() / f.groupby('depot').vol.sum()).round(4).to_dict())
sh = f.assign(q=f.date.dt.to_period('Q')).groupby(['depot', 'q']).apply(lambda g: g.ch.sum() / g.vol.sum()).unstack(0)
print(sh.round(3).T)

temp_requirement   ambient  chilled
brand                              
Fresh             107906.0  62453.0
Style              26045.0      0.0
Tech                5963.0      0.0

chilled share of Fresh volume by depot: {'Kandy': 0.3634, 'Peliyagoda': 0.3683}
q           2024Q1  2024Q2  2024Q3  2024Q4  2025Q1  2025Q2  2025Q3  2025Q4  2026Q1
depot                                                                             
Kandy        0.364   0.374   0.364   0.353   0.360   0.374   0.368   0.355   0.359
Peliyagoda   0.367   0.380   0.370   0.360   0.366   0.379   0.372   0.358   0.363


**Finding.** Style/Tech have no chilled orders (assert passes) so their chilled target is exactly 0. The Fresh chilled share is very stable over time (about 0.36-0.38 per depot), so chilled = total x share is justified and tested against a direct chilled model in 05.

## 3. Correcting the missing `not_run` orders in the test file

In [4]:
nrs = not_run_share(o)
print('train not_run share of volume per series:'); print(nrs.round(4))
raw_tot = o.groupby(['depot', 'brand']).order_volume_m3.sum()
pan_tot = panel.groupby(['depot', 'brand']).vol.sum()
print('\npanel total / raw total (>1 only through the gross-up of the 2026-02-16+ days):'); print((pan_tot / raw_tot).round(4))
rawwk = o[o.date >= '2026-02-16'].groupby(['depot', 'brand']).order_volume_m3.sum()
corr = panel[panel.date >= '2026-02-16'].groupby(['depot', 'brand']).vol.sum()
print('\nvolume added on the test-period days (m3):'); print((corr - rawwk).round(2))

train not_run share of volume per series:
depot       brand
Kandy       Fresh    0.0002
            Style    0.0134
            Tech     0.0258
Peliyagoda  Fresh    0.0061
            Style    0.0000
            Tech     0.0000
Name: order_volume_m3, dtype: float64

panel total / raw total (>1 only through the gross-up of the 2026-02-16+ days):
depot       brand
Kandy       Fresh    1.0000
            Style    1.0007
            Tech     1.0013
Peliyagoda  Fresh    1.0003
            Style    1.0000
            Tech     1.0000
dtype: float64

volume added on the test-period days (m3):
depot       brand
Kandy       Fresh     0.59
            Style     6.86
            Tech      3.13
Peliyagoda  Fresh    36.21
            Style     0.00
            Tech      0.00
dtype: float64


The gross-up is small (largest for Kandy Tech, 2.6%) and applied **only** to days where the source file cannot contain `not_run` orders. It is derived from train, so no test labels are used.

## 4. Reconciliation checks

In [5]:
# no operating day is lost or duplicated; panel covers every operating day up to the last observed day
ops = cal2[(cal2.is_operating == 1) & (cal2.date <= o.date.max())]
assert all(len(panel[(panel.depot == d) & (panel.brand == b)]) == len(ops) for d, b in SERIES)
assert panel.groupby(['depot', 'brand', 'date']).size().max() == 1
assert (panel.date.dt.dayofweek != 6).all() and panel.is_operating.eq(1).all()
# weekly totals equal the sum of the raw orders (before the gross-up) on untouched days
untouched = panel[panel.date < '2026-02-16'].groupby(['depot', 'brand']).vol.sum()
raw_u = o[o.date < '2026-02-16'].groupby(['depot', 'brand']).order_volume_m3.sum()
assert np.allclose(untouched, raw_u), 'panel does not reconcile with raw orders'
print('panel rows:', panel.shape, '| all reconciliation checks passed')
panel.to_pickle(INTERIM / 'daily_panel.pkl')
wk = panel.groupby(['depot', 'brand', 'iso_year', 'iso_week']).agg(total_m3=('vol', 'sum'), chilled_m3=('ch', 'sum'), operating_days=('date', 'size')).reset_index()
wk.to_csv(INTERIM / 'weekly_targets.csv', index=False); print('weekly targets:', wk.shape); wk.tail(6)

panel rows:

 (4170, 36) | all reconciliation checks passed
weekly targets: (702, 7)


,depot,brand,iso_year,iso_week,total_m3,chilled_m3,operating_days
696,Peliyagoda,Tech,2026,8,23.253,0.0,6
697,Peliyagoda,Tech,2026,9,40.091,0.0,6
698,Peliyagoda,Tech,2026,10,22.986,0.0,6
699,Peliyagoda,Tech,2026,11,21.290,0.0,6
700,Peliyagoda,Tech,2026,12,33.188,0.0,6
701,Peliyagoda,Tech,2026,13,37.519,0.0,6


## Decisions recorded
- Targets are built **daily on operating days** and summed; weekly series are a by-product.
- All orders count; week by `order_date`; chilled only for Fresh.
- Test-period `not_run` gap corrected with a train-derived factor (<= 2.6%).
- Leakage rule for all later work: a feature may be used only if it is known for the *future* date from `calendar.csv` (weekday, payday, festival ramp, holidays, festival distance). No lagged demand is used by the production model, so the 10-week horizon needs no recursive forecasting.